In [2]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.4/153.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 22.0 MB/s eta 0:00:00


In [3]:
import ccxt
import time
from datetime import datetime

# ⚠️ 只能用 coinbase！Colab 跑不了 Binance/OKX/Bybit
exchange = ccxt.coinbase({
    'rateLimit': 1000,
    'enableRateLimit': True,
})

symbol = 'BTC/USD'

def get_mid_price(exchange, symbol):
    """获取当前中间价"""
    ob = exchange.fetch_order_book(symbol, limit=1)
    mid = (ob['bids'][0][0] + ob['asks'][0][0]) / 2
    return mid, ob['bids'][0][0], ob['asks'][0][0]

print(f"{'='*60}")
print(f"  订单簿流向分析 (Order Book Flow)")
print(f"  {symbol} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

# 抓深度前10档
ob = exchange.fetch_order_book(symbol, limit=10)

mid_price = (ob['bids'][0][0] + ob['asks'][0][0]) / 2
print(f"\n当前中间价: ${mid_price:,.2f}")

# 模拟做市商策略：在中间价±0.1%挂单
spread_pct = 0.001  # 0.1%价差
bid_price = mid_price * (1 - spread_pct/2)
ask_price = mid_price * (1 + spread_pct/2)

print(f"做市策略: 在中间价{spread_pct*100}%范围内报价")
print(f"  买单报价: ${bid_price:,.2f}")
print(f"  卖单报价: ${ask_price:,.2f}")
print(f"  理论价差: ${ask_price - bid_price:,.2f} ({(ask_price-bid_price)/bid_price*10000:.2f} bp)")

# 分析订单簿不平衡 (Order Book Imbalance)
bid_volume_10 = sum([vol for price, vol in ob['bids']])
ask_volume_10 = sum([vol for price, vol in ob['asks']])
imbalance = (bid_volume_10 - ask_volume_10) / (bid_volume_10 + ask_volume_10) * 100

print(f"\n📊 订单簿不平衡分析:")
print(f"  买方总深度(前10档): {bid_volume_10:.4f} BTC")
print(f"  卖方总深度(前10档): {ask_volume_10:.4f} BTC")
print(f"  不平衡度: {imbalance:+.2f}%")
if imbalance > 10:
    print(f"  信号: 🟢 买方占优 (不平衡度>{'+'}10%)")
elif imbalance < -10:
    print(f"  信号: 🔴 卖方占优 (不平衡度<{'-'}10%)")
else:
    print(f"  信号: ⚪ 多空相对均衡")

# 模拟：如果做市，一次被吃单的盈亏
print(f"\n💡 做市商入门策略:")
print(f"  Step 1: 挂$限价买单 @ ${bid_price:,.2f} (低于中间价{spread_pct*100/2}%)")
print(f"  Step 2: 挂限价卖单 @ ${ask_price:,.2f} (高于中间价{spread_pct*100/2}%)")
print(f"  Step 3: 价格来回波动时，两边都被吃 = 吃一次价差")
print(f"  Step 4: 一次完整循环赚: ${ask_price-bid_price:,.2f} per BTC")
print(f"  Step 5: ⚠️ 风险：价格单边跌破买单 = 被套！必须设止损")
print(f"\n🧪 Try it yourself:")
print(f"  1. 换成 ETH/BTC 或 ETH/USD，看价差差多少")
print(f"  2. 把spread_pct从0.001改成0.0005（更窄的价差），看看能不能成功挂单")
print(f"  3. 真正的做市商策略不是设固定价差——而是根据订单簿不平衡动态调整报价")
print(f"  4. 【明日预告】我们明天用这个逻辑做订单簿不平衡预测！")

print(f"{'='*60}")

  订单簿流向分析 (Order Book Flow)
  BTC/USD | 2026-07-02 05:01:28

当前中间价: $60,617.07
做市策略: 在中间价0.1%范围内报价
  买单报价: $60,586.77
  卖单报价: $60,647.38
  理论价差: $60.62 (10.01 bp)

📊 订单簿不平衡分析:
  买方总深度(前10档): 0.9952 BTC
  卖方总深度(前10档): 0.0635 BTC
  不平衡度: +88.00%
  信号: 🟢 买方占优 (不平衡度>+10%)

💡 做市商入门策略:
  Step 1: 挂$限价买单 @ $60,586.77 (低于中间价0.05%)
  Step 2: 挂限价卖单 @ $60,647.38 (高于中间价0.05%)
  Step 3: 价格来回波动时，两边都被吃 = 吃一次价差
  Step 4: 一次完整循环赚: $60.62 per BTC
  Step 5: ⚠️ 风险：价格单边跌破买单 = 被套！必须设止损

🧪 Try it yourself:
  1. 换成 ETH/BTC 或 ETH/USD，看价差差多少
  2. 把spread_pct从0.001改成0.0005（更窄的价差），看看能不能成功挂单
  3. 真正的做市商策略不是设固定价差——而是根据订单簿不平衡动态调整报价
  4. 【明日预告】我们明天用这个逻辑做订单簿不平衡预测！


In [4]:
import ccxt
from datetime import datetime

exchange = ccxt.coinbase({'rateLimit': 1000, 'enableRateLimit': True})

# 改这里 ↓
symbol = 'ETH/USD'

# 抓深度
ob = exchange.fetch_order_book(symbol, limit=10)
mid_price = (ob['bids'][0][0] + ob['asks'][0][0]) / 2
print(f"当前中间价: ${mid_price:.2f}")

# 改这里 ↓ 价差从0.1%缩到0.05%
spread_pct = 0.0005
bid_price = mid_price * (1 - spread_pct/2)
ask_price = mid_price * (1 + spread_pct/2)

print(f"买单: ${bid_price:.2f} | 卖单: ${ask_price:.2f}")
print(f"价差: {(ask_price-bid_price)/bid_price*10000:.2f} bp")

# 买卖不平衡
bid_vol = sum([v for p,v in ob['bids']])
ask_vol = sum([v for p,v in ob['asks']])
imb = (bid_vol - ask_vol) / (bid_vol + ask_vol) * 100
print(f"\n不平衡度: {imb:+.2f}%")
print(f"买方深度: {bid_vol:.4f} | 卖方深度: {ask_vol:.4f}")

当前中间价: $1628.47
买单: $1628.07 | 卖单: $1628.88
价差: 5.00 bp

不平衡度: +48.42%
买方深度: 31.7098 | 卖方深度: 11.0192


In [7]:
import ccxt
from datetime import datetime

exchange = ccxt.coinbase({'rateLimit': 1000, 'enableRateLimit': True})

symbol = 'ETH/BTC'

ob = exchange.fetch_order_book(symbol, limit=10)
mid_price = (ob['bids'][0][0] + ob['asks'][0][0]) / 2
print(f"ETH/BTC 中间价: {mid_price:.6f} BTC")
print(f"买一: {ob['bids'][0][0]:.6f} | 卖一: {ob['asks'][0][0]:.6f}")

spread_pct = 0.0005
bid_price = mid_price * (1 - spread_pct/2)
ask_price = mid_price * (1 + spread_pct/2)
spread_bp = (ask_price - bid_price) / bid_price * 10000
print(f"价差: {spread_bp:.2f} bp")

# 深度对比
bid_vol = sum([v for p,v in ob['bids']])
ask_vol = sum([v for p,v in ob['asks']])
imb = (bid_vol - ask_vol) / (bid_vol + ask_vol) * 100
print(f"买方深度: {bid_vol:.4f} ETH")
print(f"卖方深度: {ask_vol:.4f} ETH")
print(f"不平衡度: {imb:+.2f}%")

ETH/BTC 中间价: 0.026861 BTC
买一: 0.026860 | 卖一: 0.026863
价差: 5.00 bp
买方深度: 16.7247 ETH
卖方深度: 14.4841 ETH
不平衡度: +7.18%
